# 05 — Presentation Results

Generate all publication-quality figures and summary tables for the hackathon
presentation. Everything saves to `../results/`.

Figures produced:
1. Pipeline comparison bar chart (per patient)
2. Confusion matrices for best classifier
3. Topographic ERD maps (mu + beta)
4. Laterality index comparison
5. CSP spatial patterns
6. Summary results table

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import mne
from sklearn.model_selection import cross_val_predict, StratifiedKFold

from src.loading import load_all_patients, CH_NAMES
from src.classifiers import build_all_pipelines, evaluate_all
from src.evaluation import compare_to_baseline, full_evaluation
from src.lateralization import compute_laterality_index
from src.visualization import (
    plot_pipeline_comparison,
    plot_confusion_matrix,
    plot_topographic_erd,
    plot_laterality_comparison,
    plot_csp_patterns,
)

mne.set_log_level('WARNING')
%matplotlib inline

## 5.1 Load and Classify

In [ ]:
patient_data = load_all_patients('../data/')
pipelines = build_all_pipelines()

all_results = {}
for pid, (X, y, epochs) in patient_data.items():
    results_df = evaluate_all(X, y, pipelines)
    results_df = compare_to_baseline(results_df)
    all_results[pid] = results_df

## 5.2 Pipeline Comparison Figures

In [ ]:
for pid, results_df in all_results.items():
    plot_pipeline_comparison(results_df, save_path=f'../results/fig_comparison_{pid}.png')

## 5.3 Confusion Matrices

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for pid, (X, y, _) in patient_data.items():
    best_name = all_results[pid].iloc[0]['pipeline']
    best_pipe = pipelines[best_name]
    
    y_pred = cross_val_predict(best_pipe, X, y, cv=cv)
    plot_confusion_matrix(
        y, y_pred,
        title=f'{pid} — {best_name}',
        save_path=f'../results/fig_cm_{pid}.png',
    )

## 5.4 Topographic ERD Maps

In [ ]:
for pid, (X, y, epochs) in patient_data.items():
    plot_topographic_erd(
        epochs, epochs.info['sfreq'], CH_NAMES,
        band=(8, 13),
        save_path=f'../results/fig_topo_mu_{pid}.png',
    )
    plot_topographic_erd(
        epochs, epochs.info['sfreq'], CH_NAMES,
        band=(13, 30),
        save_path=f'../results/fig_topo_beta_{pid}.png',
    )

## 5.5 Laterality Index

In [ ]:
li_results = {}
for pid, (X, y, epochs) in patient_data.items():
    li_results[pid] = compute_laterality_index(X, epochs.info['sfreq'])

plot_laterality_comparison(li_results, save_path='../results/fig_laterality.png')

## 5.6 CSP Patterns

In [ ]:
from mne.decoding import CSP

for pid, (X, y, epochs) in patient_data.items():
    csp = CSP(n_components=4, reg='ledoit_wolf', log=True)
    csp.fit(X, y)
    plot_csp_patterns(
        csp, epochs.pick('eeg').info,
        save_path=f'../results/fig_csp_{pid}.png',
    )

## 5.7 Summary Table

In [ ]:
# Grand summary: best pipeline per patient
summary_rows = []
for pid, results_df in all_results.items():
    best = results_df.iloc[0]
    li = li_results[pid]
    summary_rows.append({
        'Patient': pid,
        'Best Pipeline': best['pipeline'],
        'Accuracy': f"{best['mean_acc']:.3f} ± {best['std_acc']:.3f}",
        'vs CSP+LDA p': f"{best.get('vs_baseline_p', 'N/A')}",
        'Mu LI': f"{li['mu_li']:+.3f}",
        'Beta LI': f"{li['beta_li']:+.3f}",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv('../results/summary.csv', index=False)
print('\nSaved to ../results/summary.csv')